# Distribution Planning Benchmark

This notebook imports saved distribution-planning runs from `results/distribution`. It is analysis-only and does not run training.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))

from mfc.visualization import (
    best_runs_by_label,
    discrete_transport_tv_bound_table,
    load_runs,
    objective_table,
    plot_distribution_comparison,
    plot_state_flow,
    plot_validation_rewards,
    runtime_table,
)

ENV = 'distribution'
RESULTS_ROOT = ROOT / 'results'
runs = load_runs(RESULTS_ROOT, env=ENV)
print(f'Loaded {len(runs)} saved runs from {RESULTS_ROOT / ENV}')

## Validation Reward

Mean validation reward over training, with one standard deviation across seeds. Higher is better because the reward is the negative mismatch and movement cost.

In [ ]:
if not runs:
    print('No saved runs yet. Run scripts/run.py before executing the analysis cells.')
else:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_validation_rewards(runs, env=ENV, horizon=5, flow='exact', ax=ax)
    ax.set_title('Distribution planning validation reward')
    plt.show()

## Initial, Target, and Learned Distributions

Bar charts comparing the uniform initial law, the fixed target law, and the terminal population distribution induced by the best saved policy for each configuration.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_distribution_comparison(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"{meta['algorithm']}, perturbation={meta['perturbation']}")
    plt.show()

## Learned Flow Over Time

State-probability trajectories under selected learned policies. These curves show whether the policy moves mass coherently toward the target or oscillates around it.

In [ ]:
for run in best_runs_by_label(runs):
    fig, ax = plt.subplots(figsize=(8, 4.5))
    plot_state_flow(run, ax=ax)
    meta = run['metadata']
    ax.set_title(f"Flow: {meta['algorithm']}, perturbation={meta['perturbation']}")
    plt.show()

## Transport Perturbation Bound

For simplex transport, the total-variation distance between the sampled perturbed law and the unperturbed law is bounded by `lambda`.

In [ ]:
tv_bounds = discrete_transport_tv_bound_table(runs)
display(tv_bounds if not tv_bounds.empty else pd.DataFrame({'message': ['No transport runs found yet.']}))

## Objective and Runtime Tables

Final validation rewards, estimated simulator budgets, and runtime summaries.

In [ ]:
display(objective_table(runs))
display(runtime_table(runs))